# 数据处理

下载和读取， 参考：<https://github.com/karpathy/nanoGPT/blob/master/data/shakespeare/prepare.py>

```python
import os
import requests

# os.getcwd()    会放到 code文件夹下面，而不是 zero_gpt文件夹下面
# __file__(当前文件路径)，会放到和当前文件同一个文件夹下，但是jupyter里不能用
input_file_path = os.path.join(os.path.dirname(os.getcwd()), 'input.txt')
if not os.path.exists(input_file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(input_file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(data_url).text)

# 网络不好的话还是自己手动下载放好吧
```

## 读取数据

In [1]:
text = open("input.txt", 'r').read()
print("length of datasets in characters: ", len(text))
print(text[:1000])

length of datasets in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hun

## 分词器构建

In [2]:
# 统计字典的字符数量
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
char_ascii_num = [ord(char) for char in chars]  # chr()和ord() ord() 把字符转成数字编码，chr() 把数字编码转回字符‌
print(char_ascii_num)
# 第一个字符应该是Line Feed（直译为“送纸”或“行推进”）。
# Line Feed (LF)，将光标移动到下一行的相同水平位置（在现代计算机中通常默认也会回到行首）。
# Carriage Return (CR, 回车，ASCII 13)
# 10是换行  32是space空格 33是！
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[10, 32, 33, 36, 38, 39, 44, 45, 46, 51, 58, 59, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122]
65


In [3]:
# 字符到数字的映射， 和数字到字符的映射
# 最简单的分词 tokenizer过程
# 这里和以前不一样了，以前会加一个 . 作为终止符
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]  # lambda函数， s表示sentence/string 把一串字符编码为整数列表
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hello world!"))
print(decode(encode("hello world!")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
hello world!


还有很多分词方法，比如：
+ [openai/tiktoken](https://github.com/openai/tiktoken)
+ [google/sentencepiece](https://github.com/google/sentencepiece)

In [4]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print(enc.n_vocab)

print(enc.encode("hello world!"))
print(enc.decode([31373, 995, 0]))

50257
[31373, 995, 0]
hello world!


## 构造数据集

In [5]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)  # torch.long，即 int64
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [6]:
# 划分训练集和验证集
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
# 限制最大输入长度，因为不可能一下子把 1115394 这整个文档的char全都送到网络里去
block_size = 8
train_data[: block_size+1] 
# 这里加1表示的是用前8个字符预测第9个字符，即：在构造的数据集样本中，输入是前8个字符，输出是要预测的下一个字符
# 由于滑动窗口的存在，这里看似是9个字符，实际上包含了8个样本，全为空不算，所以可以理解为 7个空+第一个字符→第二个字符，...  0个空+8个字符→最后一个字符
# 其实快速判断的方法就是看 输出有多少种样本，很明显，除了第一个字符之外，其余8个都可以作为输出，所以有8个样本

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
demo_x = train_data[: block_size]
demo_y = train_data[1: block_size+1]
for t in range(block_size):
    context = demo_x[:t+1]  # t从0开始 [:1] 相当于取0 
    target = demo_y[t]
    print(f"when input is [{context}], the target is [{target}]")

when input is [tensor([18])], the target is [47]
when input is [tensor([18, 47])], the target is [56]
when input is [tensor([18, 47, 56])], the target is [57]
when input is [tensor([18, 47, 56, 57])], the target is [58]
when input is [tensor([18, 47, 56, 57, 58])], the target is [1]
when input is [tensor([18, 47, 56, 57, 58,  1])], the target is [15]
when input is [tensor([18, 47, 56, 57, 58,  1, 15])], the target is [47]
when input is [tensor([18, 47, 56, 57, 58,  1, 15, 47])], the target is [58]


In [9]:
a = torch.randint(300, (8,))
a

tensor([274,  11, 176, 193, 175, 233,  69, 231])

In [30]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split, batch_size = 4):
    """
    split：数据集划分，例如：train_data/val_data
    """
    data = train_data if split=="train" else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))  # low=0, high, size(tuple) 即：在0~1115394-8 这堆数里，生成4个索引值 这4个值是随机抽的，不是连续的，和data的连续的数字映射无关
    x = torch.stack([data[i:i+block_size] for i in ix]) # 默认dim = 0，堆叠成多行
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) 
    # y等于x向右偏移1个，所以加1就行，这里和之前的makemore不同，之前是输入3个，输出1个；现在是输入3个，输出3个(输入的后2个+预测的1个，这里输入不要第一个)
    # 这个构建方法就和大模型/基于Transformer的预测是一致的了
    return x,y

xb,yb = get_batch('train')
print(f"inputs: {xb}\n{xb.shape}")
print(f"targets: {yb}\n{yb.shape}")
print("-----")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is [{context}], the target is [{target}]")

inputs: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
targets: tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
torch.Size([4, 8])
-----
when input is [tensor([24])], the target is [43]
when input is [tensor([24, 43])], the target is [58]
when input is [tensor([24, 43, 58])], the target is [5]
when input is [tensor([24, 43, 58,  5])], the target is [57]
when input is [tensor([24, 43, 58,  5, 57])], the target is [1]
when input is [tensor([24, 43, 58,  5, 57,  1])], the target is [46]
when input is [tensor([24, 43, 58,  5, 57,  1, 46])], the target is [43]
when input is [tensor([24, 43, 58,  5, 57,  1, 46, 43])], the target is [39]
when input is [tensor([44])], the target is [53]
when input is [tensor([44, 53])], the target is [5

根据[大模型实战营第二期——4. XTuner 大模型单卡低成本微调实战->1.1 增量预训练微调](https://stitch.blog.csdn.net/article/details/136100537)

```bash
data: <s>世界第一高峰是珠穆朗玛峰<s>
label: 世界第一高峰是珠穆朗玛峰<s>
# 是用输出+停止符 作为输出，即：损失计算的对象
# 输入的首字符/起始字符，是不作为输出/计算损失的对象的

# 计算损失函数的时候，就相当于
# 输入，      预测输出，
  <s> 预测下一个 世
  <s>世  预测下一个 界
  ...
# 会逐个匹配的，所以输入比输出多一个起始字符

# 但是这里直接构造的输入和输出是一样长，只要保证输入 能够对到 输出 是预测下一个即可
```

这32个独立样本被打包作为一个batch送到网络中，所以看似是4个输入长度为8的序列，但是实际构成的样本数有32个。。。

# 模型

从最简单的模型开始，
1. bigram模型 → `2_build_makemore.ipynb`里 使用词频计数作为预测的方案

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

## Bigram model

这其实就是重新实现了`2_build_makemore.ipynb`里 使用词频计数作为预测的方案，用这个词频表作为决定下一个词是谁的方案，输入是单个字符，输出也是单个字符

In [12]:
vocab_size

65

### 模型定义

In [29]:
# Module 是类，首字母大写。
# modules 不是给你继承用的类，它通常是一个包/模块命名空间。
# 不要写成 class BigramLanguageModel(nn.modules):
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # nn.Embedding(num_embeddings: int,embedding_dim: int)
        # 每个token直接从这个查询表里读取下一个token的logits（即：vocab_size个元素的逻辑值，进而计算vocab_size个元素的概率，并从概率中采样下一个字符的index）
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets= None):
        """
        这里将targets设置为可选，是为了 generate中self.forward(idx)能够正常运行
        """
        # idx和targets都是(B,T)维度的整数 int
        # 即 (batch, timestamp), (批量大小,序列长度)
        # 所以logits的shape为: (B,T,C), 这里的C就是 vocab_size，老师的解释是 channel，即：每个字符的表示维度刚好是vocab_size，即num_feature/channel
        logits = self.token_embedding_table(idx)
        # loss = F.cross_entropy(logits, targets) 
        # logits.shape(4,8,65) yb.shape(4,8) 不符合 F.cross_entropy接受的参数维度，所以需要改成下面这样
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(-1,vocab_size), targets.view(-1))
            # 老师是这么写的，很清晰，但是很麻烦
            # B,T,C = logits.shape
            # logits = logits.view(B*T,C)
            # targets = targets.view(B*T)
        return logits,loss
        
    def generate(self, idx, max_new_token):
        """
        idx: (B,T) array of indices in the current context
        这个其实和 `3_MLP_makemore.ipynb# 采样` 的实现是一样的
        """
        for _ in range(max_new_token):
            logits,loss = self.forward(idx) # logits.shape(4,8,65)
            # 注意，这里idx是(B,T)形状，例如:(4,8)就可以理解为 4个样本，每个样本的上下文长度为8
            # 而这样输入带来的输出也是(4,8,65),即4个样本，上下文长度为8，每个上下文的下一个预测结果
            # 但是我们只关注最后一个，即: 从(4,8,65)→(4,1,65)
            logits = logits[:,-1,:] 
            probs = F.softmax(logits,dim = -1) # 对 65这个维度求和计算e^x
            # 这里和以前不一样， 以前是replacement=True，这里默认不改的话是False
            idx_next = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
        return idx

目前的写法看起来有点傻，明明只需要输入前一个字符就可以预测下一个，但是这里`generate()`函数里输入了8个去预测下一个

这是因为目前是个简单的 bigram，用到的上下文只有1个字符；但是如果换成后面真正的Transformer或者更复杂的模型，这样的输入就合理了

这样做是为了保证函数的一致性

In [28]:
# m 即 model
m = BigramLanguageModel(vocab_size) 
out, loss = m(xb,yb)
print(out.shape, loss) 

demo_generate = m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=100)[0].tolist()
# 拆分一下就是：
# idx = torch.zeros((1,1), dtype=torch.long)  # batch为1，上下文长度也为1，值为0 即：换行符
# generate_result = m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=100) # 生成100个新字符
# show_one = generate_result[0].tolist() #转成列表
print(decode(demo_generate)) # 数字 映射 为对应的 字符

torch.Size([4, 8, 65]) tensor(4.6815, grad_fn=<NllLossBackward0>)

,'e&E
kvrRWjEiNwc;mX'ASnJnijArEzlNLHkklqdnopvjO.Ye!RahUADiN.Ve,;3YY3IPzaPqSFnM nFkpeRA3FmqiI;l, HE
y


之前在  **4_MLP2_makemore.ipynb## 初始化分析**中，有计算过损失的标准值
+ 这里的词表大小是65，假设每个类别/单词 都是相等概率取到，则
+ 根据损失函数的公式： $loss = -\frac{1}{N}\sum_{i=1}^Nlog(y_{i})$
+ 则这里就是
  ```python
  ```

In [23]:
import math
-(math.log(1/65))*32/32  # 4.174387269895637
# 得到的结果和上面的初始loss = 4.6630 有一定的差距，说明上面的网络初始预测的分布并不均匀

4.174387269895637

In [16]:
yb.shape
# logits.shape 明显是 (4,8,65)
# 所以这里需要把 对维度进行转换

torch.Size([4, 8])

(B,T,C)维度， `B(batch size) = 4`批量大小是4, `T(timestamp) = 8`序列长度是8，`C(vocab_size) = 65`表示词表大小
+ 所以其实是对`4*8=32`个`timestamp`都预测下一个词的index
+ 而不是单纯预测`4`个样本的第8个字符后的下一个词的index

这个模型的特征就是：
+ 每个字符只能看到自己，看不到任何其他的上下文，即：输入的4个样本的8个字符是没有上下文的，都是单个字符直接去预测下一个~

----

这里由于 `F.cross_entropy`对输入和目标的维度规定，所以无法直接`loss = F.cross_entropy(idx, targets)`， 根据[torch.nn.functional.cross_entropy](https://docs.pytorch.org/docs/2.13/generated/torch.nn.functional.cross_entropy.html)
```bash
C = number of classes
N = batch size
input: shape(C),(N,C),(N,C,d1,d2,...,dk), 其中如果是K维loss，K>=1
Target: 如果包含indices，shape(),(N),(N,d1,d2,...,dk)

所以可以发现，F.cross_entropy的目标不能接受 one-hot，必须是一个样本一个类别值这样精确的标签
```

In [14]:
m.token_embedding_table.weight.shape, xb.shape

(torch.Size([65, 65]), torch.Size([4, 8]))

### 训练和评测

In [36]:
m = BigramLanguageModel(vocab_size) 

In [40]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)
# 一般这个优化器的学习率是 3e-4, 但是这里因为网络比较小，很简单，所以可以直接 1e-3

batch_size = 32
for steps in range(10000):
    # 这里优化的步数，可以随便设置，大概 14k能到 2.399 这个loss结果
    xb,yb = get_batch('train', batch_size)
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

    # 以前的做法  3_MLP_makemore.ipynb# 开始优化（5个单词） 里写的
    # for p in parameters:
    #     p.grad = None
    # 等价于 optimizer.zero_grad(set_to_none = True)
    # loss.backward()
    # for p in parameters:
    #     p.data-=lr*p.grad
    # 等价于  optimizer.step()
print(loss.item())

2.3990471363067627


对比：`2_build_makemore.ipynb## 评测模型效果/性能` 这里得到的损失是 `2.4241`（但是这里是27个字符的词表, 预测下一个字符的交叉熵损失应该是从 -math.log(1/27) = 3.295836866004329）

这里是从 `-(math.log(1/65)) = 4.174387269895637`优化到的`2.399`, 优化的差异还是存在的

In [43]:
-math.log(1/27)

3.295836866004329

In [42]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=400)[0].tolist()))



Who add ingere than,
ERENGoutne.
ONET:
It ushity-morinorisor m velyoulend hinl ano qus y glengep or ts wnd rever all a sor; silsor'late eve allseputyon:
RWApan whous aiten! p; tr powoubir d thie ct althabef inouas cou treales mesoterue tantoth dse th cee
MPoot oor dsthotouser aloootes yo the wo t opes'lir tonom tray t momand
Mugs s ur s ovaghalyembe'se Bu,
JUs the we?
We trs
Bloiaishth, arulldoke


比起上面直接初始化推理生成的结果，要好多了。。。至少看起来不是一群乱码了，有空格了

**这是从回车符开始预测的结果**， 已经很6了

```bash
,'e&E
kvrRWjEiNwc;mX'ASnJnijArEzlNLHkklqdnopvjO.Ye!RahUADiN.Ve,;3YY3IPzaPqSFnM nFkpeRA3FmqiI;l, HE
y
```

### 总结

Bigram这种模型，token之间没有关系，只有当前词和下一个词，其余没有什么上下文，窗口太窄

接下来的Transformer需要token之间开始交流（即：token之间可以看到上下文了, 窗口变大~）

## Transformer

# 其他

## nn.Embedding层随机初始化

嵌入层的初始化， 根据`3_MLP_makemore.ipynb`

```python
C = torch.randn((27,2))
emb = C[X]
print(f"emb.shape: {emb.shape}")
```

根据[Reference API -> torch.nn -> Embedding](https://docs.pytorch.org/docs/2.13/generated/torch.nn.Embedding.html)
```bash
Variables: weight (Tensor) – the learnable weights of the module of shape (num_embeddings, embedding_dim) initialized from N(0,1)

# 只写：nn.Embedding(num_embeddings, embedding_dim)
# PyTorch 会创建一个形状为：(num_embeddings, embedding_dim)的可训练参数 weight，并且默认会用N(mean=0, std=1)初始化
```
根据[torch/nn/modules/sparse.py](https://github.com/pytorch/pytorch/blob/v2.13.0/torch/nn/modules/sparse.py#L164-L181)
```python
if _weight is None:
    self.weight = Parameter(
        torch.empty((num_embeddings, embedding_dim), **factory_kwargs),
        requires_grad=not _freeze,
    )
    self.reset_parameters()   # 当没有对nn.Embedding加载外部训练好的权重的时候，会先创建一个空tensor(即：self.weight)，然后调用这个函数对空tensor赋值
...

def reset_parameters(self) -> None:
    init.normal_(self.weight)   # 这里就是对 self.weight真正进行初始化的地方
    self._fill_padding_idx_with_zero()

from torch.nn import functional as F, init
# 再结合  https://docs.pytorch.org/docs/2.13/nn.init.html#torch.nn.init.normal_
torch.nn.init.normal_(tensor, mean=0.0, std=1.0, generator=None)[source]
# Fill the input Tensor with values drawn from the normal distribution.
```
可知，  embedding table 中的每个元素默认来自标准正态分布，不存在以行为单位进行正态分布的采样这个说法

每个标量元素独立地从标准正态分布采样。所以，如果你说的“以行为单位进行正态分布采样”是指：
```bash
每一行有一个单独的均值、方差，或者每一行调用一次类似 normal_(dim=1) 的采样接口
```
那默认实现里没有这种说法，也没有 `dim=1` 这种参数。

## optimizer.zero_grad(set_to_none = True)

根据 [torch.optim.Optimizer.zero_grad](https://docs.pytorch.org/docs/2.13/generated/torch.optim.Optimizer.zero_grad.html)

```bash
不要将梯度设为零，而是将其设为 None。默认值：True。

这通常会减少内存占用，并能适度提升性能。但会改变某些行为。例如：

当用户尝试访问梯度并对其进行手动操作时，None属性或充满0的张量会表现出不同的行为。

如果用户先请求 zero_grad(set_to_none=True)，然后进行反向传播，对于未接收梯度的参数，.grads 保证为 None。

torch.optim 优化器在梯度为 0 或 None 时具有不同的行为（一种情况下使用梯度为 0 的步长，另一种情况下则完全跳过该步）。
```